In [ ]:

import json
import os

from pathlib import Path
from urllib.request import urlopen, Request
from openai import OpenAI
from typing import Optional, List, Dict, Any
from pydantic import BaseModel

SHORT_CITATION_SYSTEM_PROMPT = """## System Instruction: Short Citation Generator

You are given a full academic reference as input. Your task is to convert it into the shortest possible recognizable citation format.

### Output format
FirstAuthorLastName et al., JournalAbbreviation, Year

### Rules
1. Extract the **last name of the first author only**.
2. Always append **“et al.”** after the first author.
3. Extract the **journal name** and convert it to its **standard abbreviation** (e.g., Scientific Reports → Sci Rep).
4. Extract the **publication year only**.
5. Ignore all other information (title, volume, issue, pages, DOI, PMID, etc.).
6. Use this exact punctuation and spacing:
   - Comma after “et al.”
   - Comma before the year
   - No period at the end

### Additional parsing rules for structured inputs
- If multiple `AU` fields exist, use the **first AU entry** as the first author.
- Prefer `J2` for the journal abbreviation. If not available, abbreviate `JF`.
- Extract the year from any date field (e.g., `Y1  - YYYY/...` → use YYYY only).

### Example

**Input:**
AU  - Dolgalev, Igor
AU  - Zhou, Hua
Y1  - 2023/11/08
J2  - Nat Commun

**Output:**
Dolgalev et al., Nat Commun, 2023

### Constraint
Output only the formatted citation. Do not include explanations or additional text.
"""


# gets API Key from environment variable OPENAI_API_KEY
client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY")
)


class ResponseFormat(BaseModel):
    response: str


def call_openai(
    user_text: str,
    *,
    system_instructions: Optional[str] = None,
    model: str = "gpt-5-nano-2025-08-07",
    verbosity: str = "low",
    reasoning_effort: str = "high",
) -> str:
    messages: List[Dict[str, Any]] = []
    if system_instructions:
        messages.append(
            {
                "role": "system",
                "content": [{"type": "input_text", "text": system_instructions}],
            }
        )
    messages.append(
        {
            "role": "user",
            "content": [{"type": "input_text", "text": user_text}],
        }
    )

    resp = client.responses.parse(
        model=model,
        input=messages,
        text={"verbosity": verbosity},
        reasoning={"effort": reasoning_effort},
        service_tier="flex",
        text_format=ResponseFormat,
    )
    return resp.output_parsed


def pmcid_to_short_citation(pmcid: str) -> str:
    if not pmcid.startswith("PMC"):
        raise ValueError(f"Invalid PMCID: {pmcid!r}")
    
    pmc_numeric_id = pmcid.replace("PMC", "")

    ris_url = f"https://api.ncbi.nlm.nih.gov/lit/ctxp/v1/pmc/?id={pmc_numeric_id}&format=ris"
    req = Request(ris_url)
    with urlopen(req) as r:
        ris_text = r.read().decode("utf-8", errors="replace")

    result = call_openai(
        ris_text,
        system_instructions=SHORT_CITATION_SYSTEM_PROMPT,
    )

    # Your call_openai currently returns resp.output_parsed; handle both shapes.
    if hasattr(result, "response"):
        return {'pmcid': pmcid, 'citation': str(result.response).strip()}
    return {'pmcid': pmcid, 'citation': str(result).strip()}



project_root = Path().resolve().parent.parent.parent
publish_metadata_dir = project_root / "publish" / "metadata"
publish_metadata_dir.mkdir(parents=True, exist_ok=True)
summary_path = publish_metadata_dir / "data_summary.json"
annotation_file_path = publish_metadata_dir / "pmcid_to_citation.json"

with open(summary_path, "r") as f:
    data_summary = json.load(f)

# gse_ids = [item["data_id"] for item in data_summary if item.get("data_id", "").startswith("GSE")]
pmcids = list(set(pmcid for item in data_summary for pmcid in item["pmcids"]))
print(f"Found {len(pmcids)} PMC IDs")

# Output is a JSON array of annotation objects.
if annotation_file_path.exists():
    with open(annotation_file_path, "r") as f:
        existing_results = json.load(f)
    if not isinstance(existing_results, list):
        raise ValueError("Expected existing annotation file to contain a JSON array")
else:
    existing_results = []

existing_ids = {
    item.get("pmcid")
    for item in existing_results
    if isinstance(item, dict) and item.get("pmcid")
}

results = list(existing_results)

for pmcid in pmcids:
    if pmcid in existing_ids:
        print(f"Skipping {pmcid} (already exists)")
        continue

    print(f"Generating annotation for {pmcid}...")
    result = pmcid_to_short_citation(pmcid)
    results.append(result)
    existing_ids.add(pmcid)

    # Save after each item so partial progress is kept.
    with open(annotation_file_path, "w") as f:
        json.dump(results, f, indent=4)

    print(f"Saved {pmcid}")

print(f"Done. Total saved annotations: {len(results)}")


Found 42 PMC IDs
Skipping PMC7446376 (already exists)
Skipping PMC9247075 (already exists)
Skipping PMC11669362 (already exists)
Skipping PMC10632519 (already exists)
Skipping PMC9780475 (already exists)
Skipping PMC11297740 (already exists)
Skipping PMC8421377 (already exists)
Skipping PMC10902356 (already exists)
Skipping PMC7339020 (already exists)
Skipping PMC10071545 (already exists)
Skipping PMC9972164 (already exists)
Skipping PMC10226260 (already exists)
Skipping PMC8391231 (already exists)
Skipping PMC5701046 (already exists)
Skipping PMC8800070 (already exists)
Skipping PMC7338507 (already exists)
Skipping PMC7335061 (already exists)
Skipping PMC6971301 (already exists)
Skipping PMC7839060 (already exists)
Skipping PMC6704148 (already exists)
Skipping PMC6206135 (already exists)
Skipping PMC5704234 (already exists)
Skipping PMC8484871 (already exists)
Skipping PMC8718409 (already exists)
Skipping PMC9450692 (already exists)
Skipping PMC10192868 (already exists)
Skipping PMC94